In [ ]:
# -----------------------------------------------------------------------------
# # Garman-Kohlhagen (Black-Scholes for FX): price + full Greek set
# -----------------------------------------------------------------------------

def gk_price(S, K, T, rd, rf, sigma, is_call=True):
    '''Garman-Kohlhagen price. S=spot, K=strike, T=year fraction, rd=domestic rate,
    rf=foreign rate, sigma=vol (decimal, e.g. 0.08 = 8%).'''
    vst = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma ** 2) * T) / vst
    d2 = d1 - vst
    Dq, Dd = np.exp(-rf * T), np.exp(-rd * T)
    if is_call:
        return S * Dq * norm.cdf(d1) - K * Dd * norm.cdf(d2)
    return K * Dd * norm.cdf(-d2) - S * Dq * norm.cdf(-d1)


def gk_greeks(S, K, T, rd, rf, sigma):
    '''Delta (call & put), gamma, vega, vanna, volga. Closed forms (Haug, Complete Guide to
    Option Pricing Formulas); vanna/volga computed at flat sigma and are option-type-invariant.'''
    vst = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (rd - rf + 0.5 * sigma ** 2) * T) / vst
    d2 = d1 - vst
    Dq = np.exp(-rf * T)
    pdf1 = norm.pdf(d1)
    delta_call = Dq * norm.cdf(d1)
    delta_put = delta_call - Dq
    gamma = Dq * pdf1 / (S * vst)
    vega = S * Dq * pdf1 * np.sqrt(T)          # per 1.0 (=100%) vol move
    vanna = -Dq * pdf1 * d2 / sigma            # d(vega)/dS == d(delta)/d(sigma)
    volga = vega * d1 * d2 / sigma             # d(vega)/d(sigma), aka "vomma"
    return dict(d1=d1, d2=d2, delta_call=delta_call, delta_put=delta_put,
                gamma=gamma, vega=vega, vanna=vanna, volga=volga)


def atm_strike(S, T, rd, rf, sigma_atm):
    '''Delta-neutral-straddle ATM strike: K such that d1 = 0.'''
    F = S * np.exp((rd - rf) * T)
    return F * np.exp(0.5 * sigma_atm ** 2 * T)


def delta_to_strike(S, T, rd, rf, sigma, delta_mag, is_call):
    '''Invert plain spot delta -> strike. delta_mag in (0,1), e.g. 0.25 for "the 25-delta" option
    (call or put, sign handled by is_call).'''
    F = S * np.exp((rd - rf) * T)
    Dq = np.exp(-rf * T)
    d1 = norm.ppf(delta_mag / Dq) if is_call else -norm.ppf(delta_mag / Dq)
    return F * np.exp(-sigma * np.sqrt(T) * d1 + 0.5 * sigma ** 2 * T)


# Módulo 2

## Vanna-volga: de 3 cotizaciones a una sonrisa completa

> **Objetivo de aprendizaje.** Construir una sonrisa de volatilidad completa a partir de tres cotizaciones replicando Vega/Vanna/Volga en tres pilares líquidos, y ver exactamente dónde falla el método para que conozcas su dominio de validez antes de confiar en él.

Tres cotizaciones (ATM, RR25, BF25) definen tres puntos de la sonrisa. Vanna-volga (Castagna & Mercurio, 2007) es la técnica estándar de creación de mercado (market-making) para interpolar/extrapolar el **resto de la curva de manera consistente**. Vale la pena detenerse en ella porque utiliza la misma idea (una derivada más abajo) que la lógica de cartera de réplica empleada más adelante para la negociación eficiente en costes de una señal cara/barata: **replicar el riesgo de un instrumento arbitrario utilizando una base pequeña de instrumentos líquidos, y luego valorar el residuo.**

**Construcción.** Para un precio de ejercicio objetivo $K$, encontrar los pesos $w_1, w_2, w_3$ en las tres opciones pilares de tal manera que la cartera iguale las griegas **Vega, Vanna y Volga** del objetivo (todas evaluadas a una volatilidad plana $\sigma_{\text{ATM}}$):

$$M \mathbf{w} = \mathbf{g}(K), \quad M = \begin{pmatrix} \text{Vega}_1 & \text{Vega}_2 & \text{Vega}_3 \\ \text{Vanna}_1 & \text{Vanna}_2 & \text{Vanna}_3 \\ \text{Volga}_1 & \text{Volga}_2 & \text{Volga}_3 \end{pmatrix}$$

Entonces, el precio vanna-volga es el precio de Black-Scholes a volatilidad plana $\sigma_{\text{ATM}}$ **más** el coste de esa cesta de réplica por encima de su propio precio a volatilidad plana:

$$P_{\text{VV}}(K) = P_{\text{BS}}(K, \sigma_{\text{ATM}}) + \sum_{i} w_i \left[ P_{\text{mkt}}(K_i, \sigma_i) - P_{\text{BS}}(K_i, \sigma_{\text{ATM}}) \right]$$

Invierta $P_{\text{VV}}(K)$ de nuevo a una volatilidad implícita mediante la búsqueda de raíces de Black-Scholes. Tenga en cuenta que las diferencias de precio anteriores son **invariantes frente a opciones call/put** —por paridad call-put, el término forward se cancela, por lo que resulta irrelevante si los pilares se valoran como calls o puts en todo momento (utilizamos calls uniformemente por simplicidad).

**Una advertencia que vale la pena interiorizar.** Vanna-volga es una réplica *local*: iguala Vega/Vanna/Volga en los tres pilares cotizados, por lo que interpola bien entre ellos y extrapola de forma fiable sólo a una corta distancia más allá —los profesionales suelen confiar en ella aproximadamente en el rango de delta~5 a delta-95. Si se empuja la rejilla de strikes mucho más hacia los extremos, el sistema de pesos de 3x3 puede producir una sonrisa **no monótona, con "doble joroba"** —no se trata de un error de código, sino de que el método VV está saliendo de su dominio validado. Sembramos la rejilla de strikes a continuación en relación con la propia escala de volatilidad de la opción ($\sigma_{\text{ATM}} \sqrt{T}$) para mantenernos dentro de ese dominio; intente ampliarla usted mismo para ver aparecer este artefacto numérico.


# Módulo 3

## Vol local (Dupire), SABR, y la cuestión de sticky-strike / sticky-delta

> **Objetivo de aprendizaje.** Ver las dos formas clásicas de convertir una sonrisa en una superficie completa —vol local de ajuste exacto frente a la dinámica realista de SABR— y comprender por qué la elección de sticky-strike/sticky-delta cambia tu delta incluso cuando la sonrisa de hoy es idéntica.

La **vol local de Dupire** es la única difusión consistente con toda la superficie de volatilidad implícita actual: revaloriza cada opción vanilla exactamente, a costa de una dinámica de sonrisa forward poco realista. En términos de la varianza implícita total $w(y, T) = \sigma_{\text{BS}}^2(y, T) T$ con el log-forward-moneyness $y = \ln(K/F(T))$ (Gatheral, The Volatility Surface, eq. 1.10):

$$\sigma_{\text{loc}}^2(y, T) = \frac{\partial w / \partial T}{1 - \frac{y}{w}\frac{\partial w}{\partial y} + \frac{1}{4}\left(-\frac{1}{4} - \frac{1}{w} + \frac{y^2}{w^2}\right)\left(\frac{\partial w}{\partial y}\right)^2 + \frac{1}{2}\frac{\partial^2 w}{\partial y^2}}$$

**SABR** (Hagan et al., 2002), en cambio, postula una SDE de volatilidad estocástica y ofrece una forma cerrada asintótica para la vol de Black como función de $(\alpha, \beta, \rho, \nu)$ — 3 parámetros libres (con $\beta$ fija) coinciden exactamente con nuestros 3 pilares cotizados, y la dinámica de SABR es mucho más realista que el aplanamiento de la sonrisa forward de la vol local. Ajustamos ambos a continuación a los mismos 3 pilares y comparamos.

**Sticky strike vs sticky delta** — la pregunta práctica que decide cómo haces el re-delta de una cartera a medida que el **spot se mueve**:

* **Sticky strike:** $\sigma(K)$ es invariante respecto al spot — la sonrisa está "fijada" en el espacio de strikes. Una opción que empieza ATM se desplaza hacia dentro/fuera del dinero a medida que el spot se mueve, y su vol se mueve a lo largo de la curva congelada.
* **Sticky delta / sticky moneyness:** $\sigma(\Delta)$ (equivalentemente $\sigma(K/F)$) es invariante respecto al spot — toda la curva se desplaza con el spot, por lo que un punto fijo de moneyness mantiene la misma vol.

Bajo sticky delta, la delta consistente con la sonrisa (delta "total") adquiere un término adicional en relación con la delta de Black-Scholes de vol plana: $\Delta_{\text{tot}} = \Delta_{\text{BS}} + \text{Vega} \cdot \partial\sigma/\partial S$. Bajo sticky strike ese término adicional es cero por construcción. Esta es exactamente la ambigüedad que motiva los esquemas de cobertura delta dependientes del régimen (por ejemplo, cubrir de manera más o menos agresiva dependiendo de si el mercado se está comportando más como sticky-strike o sticky-delta) en lugar de una regla fija.
